# EfficientNet-B1 + FLAIR — 5-Fold Patient-Level CV (BraTS 2021)
## Protocol for comparison with Saeed et al. (2023), Table 4

This notebook replaces the previous **XGBoost + GLCM voxel-level** pipeline with an **image-based 3D EfficientNet-B1** pipeline.

### Target protocol
- Input: **BraTS 2021 Task 1 FLAIR NIfTI volumes** for patients having MGMT labels.
- Target: **MGMT promoter methylation status** (`0 = unmethylated`, `1 = methylated`).
- Prediction unit: **one patient**.
- Model: **3D EfficientNet-B1** using MONAI/PyTorch.
- Validation: **5-fold patient-level cross-validation**.
- Primary metric: **ROC-AUC per fold**, plus Mean AUC and sample variance (`ddof=1`) to match the reporting convention in Saeed et al. Table 4.

### Important reproducibility note
Saeed et al. (2023) explicitly report MONAI/PyTorch, AdamW, FLAIR, EfficientNet-B1 and 5-fold CV, but the paper does **not** publish every training hyperparameter for the Table 4 EfficientNet-B1 experiment. Therefore, parameters marked **implementation choice** below are not claimed to be the exact unpublished settings of the paper.


## 0. Install dependencies

Run once in a clean Colab/Kaggle environment. `EfficientNetBN(..., spatial_dims=3)` is provided by MONAI.


In [ ]:
# Uncomment if needed:
# !pip install -q "monai>=1.3" nibabel scikit-learn pandas matplotlib


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/workspaces/todo/.venv/lib/python3.9/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspaces/todo/.venv/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/workspaces/todo/.venv/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/workspaces/todo/.venv/lib/python3.9/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
  File "/workspaces/todo/.venv/lib/python3.9/site-packages/traitlets/traitlets.py", line 668, in get
    value = self._validate(obj, default)
  File "/workspaces/todo/.venv/lib/python3.9/s

In [2]:
import os
import re
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    balanced_accuracy_score, matthews_corrcoef
)

from monai.networks.nets import EfficientNetBN

SEED = 42
N_SPLITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


2026-09-29 16:40:59.185131: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-29 16:41:02.801920: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-29 16:41:02.802481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-29 16:41:03.084450: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-29 16:41:03.738800: I tensorflow/core/platform/cpu_feature_guar

Torch: 2.8.0+cu128
Device: cuda
GPU: NVIDIA GeForce GTX 1660 SUPER


## 1. Paths and training configuration

Expected Task 1 structure is similar to:

```text
BraTS2021_Training_Data/
  BraTS2021_00000/
    BraTS2021_00000_flair.nii.gz
  ...
train_labels.csv
```

`train_labels.csv` should contain a patient ID column such as `BraTS21ID` and an MGMT label column such as `MGMT_value`.

The default `256 × 256 × 64` 3D input and LR/epoch choices are **implementation settings informed by the earlier Saeed et al. experimental protocol**, not parameters explicitly reported for Table 4 of the 2023 paper. If GPU memory is insufficient, switch `PROFILE` to `memory_safe`; do not claim that memory-safe results reproduce the paper configuration.


In [3]:
# ---------- EDIT THESE TWO PATHS ----------
BRATS_TASK1_ROOT = Path("dataset/BraTS2021_Training_Data")
MGMT_LABELS_CSV = Path("train_labels.csv")

OUTPUT_DIR = Path("output/efficientnet_b1_flair_mgmt_5fold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# "paper_like": intended for a large-memory GPU.
# "memory_safe": smaller input/batch for limited GPU memory; less directly comparable.
PROFILE = "paper_like"

PROFILES = {
    "paper_like": {
        "spatial_size_dhw": (64, 256, 256),
        "batch_size": 4,
        "epochs": 30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-2,
    },
    "memory_safe": {
        "spatial_size_dhw": (48, 128, 128),
        "batch_size": 1,
        "epochs": 30,
        "learning_rate": 1e-3,
        "weight_decay": 1e-2,
    },
}

CFG = PROFILES[PROFILE].copy()
CFG.update({
    "model": "3D EfficientNet-B1",
    "modality": "FLAIR",
    "optimizer": "AdamW",
    "loss": "BCEWithLogitsLoss",
    "n_splits": N_SPLITS,
    "seed": SEED,
    "num_workers": 2,
    "use_amp": False,       # implementation optimization; not reported by Saeed et al.
    "augmentation": False,  # keep disabled for a simple paper-aligned baseline
})

print(json.dumps(CFG, indent=2))
with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)


{
  "spatial_size_dhw": [
    64,
    256,
    256
  ],
  "batch_size": 4,
  "epochs": 30,
  "learning_rate": 0.001,
  "weight_decay": 0.01,
  "model": "3D EfficientNet-B1",
  "modality": "FLAIR",
  "optimizer": "AdamW",
  "loss": "BCEWithLogitsLoss",
  "n_splits": 5,
  "seed": 42,
  "num_workers": 2,
  "use_amp": false,
  "augmentation": false
}


## 2. Build the patient manifest: Task 1 FLAIR + MGMT label

The paper reports that the radiogenomic cohort is a subset of segmentation patients for whom MGMT labels are available. It also reports excluding IDs `00109`, `00123`, and `00709` due to unacceptable image quality. This notebook applies the same exclusions when those IDs are present.


In [4]:
BAD_IDS = {"00109", "00123", "00709"}

ID_CANDIDATES = ["BraTS21ID", "patient_id", "PatientID", "BraTS_ID", "id"]
LABEL_CANDIDATES = ["MGMT_value", "MGMT", "label", "target"]

def normalize_pid(x):
    s = str(x).strip()
    # Handles numeric IDs, 00000, BraTS2021_00000, etc.
    digits = re.findall(r"\d+", s)
    if not digits:
        return s
    return digits[-1].zfill(5)

def find_flair_files(root: Path):
    patterns = ["**/*flair.nii.gz", "**/*FLAIR.nii.gz", "**/*flair.nii", "**/*FLAIR.nii"]
    files = []
    for pat in patterns:
        files.extend(root.glob(pat))
    files = sorted(set(files))
    mapping = {}
    for p in files:
        # Prefer the parent folder, then filename, for patient-ID extraction.
        pid = normalize_pid(p.parent.name)
        if pid == p.parent.name:
            pid = normalize_pid(p.name)
        mapping[pid] = p
    return mapping

labels = pd.read_csv(MGMT_LABELS_CSV)
id_col = next((c for c in ID_CANDIDATES if c in labels.columns), None)
label_col = next((c for c in LABEL_CANDIDATES if c in labels.columns), None)
if id_col is None or label_col is None:
    raise ValueError(
        f"Không tìm thấy cột ID/label. Columns hiện có: {labels.columns.tolist()}"
    )

labels = labels[[id_col, label_col]].copy()
labels["patient_id"] = labels[id_col].map(normalize_pid)
labels["mgmt_label"] = pd.to_numeric(labels[label_col], errors="coerce")
labels = labels.dropna(subset=["mgmt_label"]).copy()
labels["mgmt_label"] = labels["mgmt_label"].astype(int)
labels = labels[labels["mgmt_label"].isin([0, 1])]
labels = labels[~labels["patient_id"].isin(BAD_IDS)]

flair_map = find_flair_files(BRATS_TASK1_ROOT)
manifest = labels[labels["patient_id"].isin(flair_map)].copy()
manifest["flair_path"] = manifest["patient_id"].map(lambda x: str(flair_map[x]))
manifest = manifest.drop_duplicates("patient_id").sort_values("patient_id").reset_index(drop=True)

missing = sorted(set(labels["patient_id"]) - set(manifest["patient_id"]))

print("MGMT labels after exclusions:", len(labels))
print("Matched Task 1 FLAIR patients:", len(manifest))
print("Missing FLAIR files:", len(missing))
print("Class distribution:")
print(manifest["mgmt_label"].value_counts().sort_index())

if len(manifest) < 5:
    raise RuntimeError("Quá ít patient được match. Hãy kiểm tra BRATS_TASK1_ROOT và ID format.")
if manifest["mgmt_label"].nunique() != 2:
    raise RuntimeError("Manifest không có đủ hai lớp MGMT 0/1.")

manifest.to_csv(OUTPUT_DIR / "patient_manifest.csv", index=False)
display(manifest.head())


FileNotFoundError: [Errno 2] No such file or directory: 'train_labels.csv'

## 3. 3D FLAIR preprocessing

For each patient:
1. Load the Task 1 FLAIR NIfTI volume.
2. Find the axial slice with the largest non-zero brain cross-section.
3. Keep a fixed depth around that center and pad if necessary.
4. Resize to the configured 3D input size.
5. Apply non-zero z-score intensity normalization.

Steps 2 and the `256×256×64` target are implementation choices derived from the earlier experimental description by the same group; the 2023 Table 4 does not fully specify these details. The z-score normalization is an explicit notebook implementation choice.


In [ ]:
def center_crop_depth(vol_hwd: np.ndarray, target_d: int) -> np.ndarray:
    # NIfTI is handled here as H x W x D.
    brain_area = (vol_hwd != 0).sum(axis=(0, 1))
    if brain_area.max() == 0:
        center = vol_hwd.shape[2] // 2
    else:
        center = int(np.argmax(brain_area))

    start = center - target_d // 2
    end = start + target_d

    src_start = max(start, 0)
    src_end = min(end, vol_hwd.shape[2])
    cropped = vol_hwd[:, :, src_start:src_end]

    pad_before = max(0, -start)
    pad_after = max(0, end - vol_hwd.shape[2])
    if pad_before or pad_after:
        cropped = np.pad(cropped, ((0, 0), (0, 0), (pad_before, pad_after)), mode="constant")
    return cropped


def load_flair_tensor(path: str, spatial_size_dhw=(64, 256, 256)) -> torch.Tensor:
    vol = nib.load(path).get_fdata(dtype=np.float32)
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D NIfTI, got shape={vol.shape}: {path}")

    target_d, target_h, target_w = spatial_size_dhw
    vol = center_crop_depth(vol, target_d)

    # Non-zero z-score normalization (explicit implementation choice).
    nz = vol != 0
    if nz.any():
        mean = float(vol[nz].mean())
        std = float(vol[nz].std())
        if std > 1e-6:
            vol[nz] = (vol[nz] - mean) / std
        else:
            vol[nz] = vol[nz] - mean

    # H,W,D -> D,H,W -> C,D,H,W
    x = torch.from_numpy(np.transpose(vol, (2, 0, 1))).unsqueeze(0).float()
    x = F.interpolate(
        x.unsqueeze(0),
        size=(target_d, target_h, target_w),
        mode="trilinear",
        align_corners=False,
    ).squeeze(0)
    return x

# Smoke-check one patient.
x0 = load_flair_tensor(manifest.iloc[0]["flair_path"], CFG["spatial_size_dhw"])
print("Example tensor C,D,H,W:", tuple(x0.shape))
print("Label:", int(manifest.iloc[0]["mgmt_label"]))


## 4. Dataset and DataLoader

There is **one sample per patient**, so patient leakage cannot occur between train and validation indices within a fold.


In [ ]:
class BraTSMGMTDataset(Dataset):
    def __init__(self, df: pd.DataFrame, spatial_size_dhw):
        self.df = df.reset_index(drop=True).copy()
        self.spatial_size_dhw = spatial_size_dhw

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_flair_tensor(row["flair_path"], self.spatial_size_dhw)
        label = torch.tensor(float(row["mgmt_label"]), dtype=torch.float32)
        return {
            "image": image,
            "label": label,
            "patient_id": row["patient_id"],
        }


def make_loader(df, shuffle):
    ds = BraTSMGMTDataset(df, CFG["spatial_size_dhw"])
    return DataLoader(
        ds,
        batch_size=CFG["batch_size"],
        shuffle=shuffle,
        num_workers=CFG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(CFG["num_workers"] > 0),
    )


## 5. 3D EfficientNet-B1

MONAI supports EfficientNet-B1 with `spatial_dims=3`. Pretrained ImageNet weights are not available for the 3D variant; this notebook therefore uses **random initialization**.


In [ ]:
def build_model():
    model = EfficientNetBN(
        model_name="efficientnet-b1",
        pretrained=False,
        spatial_dims=3,
        in_channels=1,
        num_classes=1,
    )
    return model.to(DEVICE)

model = build_model()
params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {params:,}")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. Train/evaluate functions

- Optimizer: **AdamW** (reported by Saeed et al. 2023).
- Loss: **binary cross-entropy with logits**, consistent with the paper's BCE analysis.
- Primary evaluation: **patient-level ROC-AUC**.

The learning rate, weight decay and epoch count below are implementation settings because Table 4 does not publish them explicitly.


In [ ]:
def binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    ys, ps, pids = [], [], []

    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        amp_enabled = bool(CFG["use_amp"] and DEVICE.type == "cuda")
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(x).reshape(-1)
            loss = criterion(logits, y)

        if training:
            if scaler is not None and amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        prob = torch.sigmoid(logits)
        total_loss += float(loss.item()) * len(y)
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(prob.detach().cpu().numpy().tolist())
        pids.extend(list(batch["patient_id"]))

    result = {
        "loss": total_loss / max(len(loader.dataset), 1),
        "y_true": np.asarray(ys, dtype=int),
        "y_prob": np.asarray(ps, dtype=float),
        "patient_id": pids,
    }
    if len(np.unique(result["y_true"])) == 2:
        result.update(binary_metrics(result["y_true"], result["y_prob"]))
    return result


## 7. Five-fold patient-level CV

`StratifiedKFold` is used here because there is exactly one row per patient and the target is binary. Saeed et al. report **5-fold CV** but do not publish the exact fold assignment, so this notebook cannot reproduce their fold membership exactly.

The best checkpoint within each fold is selected by that fold's validation ROC-AUC. The selected validation AUC becomes the reported fold AUC.


In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_rows = []
fold_assignments = []
oof_rows = []

X_idx = np.arange(len(manifest))
y_all = manifest["mgmt_label"].to_numpy()

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_idx, y_all), start=1):
    train_df = manifest.iloc[tr_idx].reset_index(drop=True)
    val_df = manifest.iloc[va_idx].reset_index(drop=True)

    overlap = set(train_df["patient_id"]) & set(val_df["patient_id"])
    if overlap:
        raise RuntimeError(f"Fold {fold}: patient leakage detected: {len(overlap)}")

    print("\n" + "=" * 80)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("Train patients:", len(train_df), train_df["mgmt_label"].value_counts().to_dict())
    print("Val patients  :", len(val_df), val_df["mgmt_label"].value_counts().to_dict())

    train_loader = make_loader(train_df, shuffle=True)
    val_loader = make_loader(val_df, shuffle=False)

    model = build_model()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["learning_rate"],
        weight_decay=CFG["weight_decay"],
    )

    # Step-based LR scheduling was used in the earlier Saeed experimental protocol.
    # These exact scheduler values were not reported for the 2023 Table 4 experiment.
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=bool(CFG["use_amp"] and DEVICE.type == "cuda"))

    best_auc = -np.inf
    best_epoch = None
    best_state = None
    history = []

    for epoch in range(1, CFG["epochs"] + 1):
        train_out = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
        with torch.no_grad():
            val_out = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None)

        val_auc = val_out.get("roc_auc", np.nan)
        history.append({
            "fold": fold,
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_out["loss"],
            "train_auc": train_out.get("roc_auc", np.nan),
            "val_loss": val_out["loss"],
            "val_auc": val_auc,
        })

        if np.isfinite(val_auc) and val_auc > best_auc:
            best_auc = float(val_auc)
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        print(
            f"Fold {fold} | Epoch {epoch:02d}/{CFG['epochs']} | "
            f"train loss={train_out['loss']:.4f} auc={train_out.get('roc_auc', np.nan):.4f} | "
            f"val loss={val_out['loss']:.4f} auc={val_auc:.4f}"
        )

    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / f"fold_{fold}_history.csv", index=False)

    if best_state is None:
        raise RuntimeError(f"Fold {fold}: no valid checkpoint was created.")

    model.load_state_dict(best_state)
    torch.save(best_state, OUTPUT_DIR / f"efficientnet_b1_fold_{fold}.pt")

    with torch.no_grad():
        best_val = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None)

    metrics = binary_metrics(best_val["y_true"], best_val["y_prob"])
    fold_rows.append({
        "fold": fold,
        "train_patients": len(train_df),
        "validation_patients": len(val_df),
        "train_mgmt_0": int((train_df["mgmt_label"] == 0).sum()),
        "train_mgmt_1": int((train_df["mgmt_label"] == 1).sum()),
        "validation_mgmt_0": int((val_df["mgmt_label"] == 0).sum()),
        "validation_mgmt_1": int((val_df["mgmt_label"] == 1).sum()),
        "best_epoch": best_epoch,
        **metrics,
    })

    for pid in train_df["patient_id"]:
        fold_assignments.append({"fold": fold, "patient_id": pid, "split": "train"})
    for pid in val_df["patient_id"]:
        fold_assignments.append({"fold": fold, "patient_id": pid, "split": "validation"})

    for pid, yt, yp in zip(best_val["patient_id"], best_val["y_true"], best_val["y_prob"]):
        oof_rows.append({
            "fold": fold,
            "patient_id": pid,
            "mgmt_label": int(yt),
            "prob_methylated": float(yp),
        })

    del model, optimizer, scheduler, train_loader, val_loader, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 8. Fold metrics, Mean AUC and Variance

Saeed et al. Table 4 reports EfficientNet-B1 + FLAIR AUCs:

`0.6128 / 0.5810 / 0.6300 / 0.6555 / 0.5546`, mean `0.6068`, variance `0.0016`.

Using sample variance (`ddof=1`) reproduces the reported variance after rounding.


In [ ]:
SAEED_AUC_FOLDS = np.array([0.6128, 0.5810, 0.6300, 0.6555, 0.5546], dtype=float)
SAEED_MEAN_AUC = 0.6068
SAEED_VARIANCE_AUC = 0.0016

fold_results = pd.DataFrame(fold_rows).sort_values("fold").reset_index(drop=True)
fold_results.to_csv(OUTPUT_DIR / "efficientnet_b1_flair_5fold_metrics.csv", index=False)

pd.DataFrame(fold_assignments).to_csv(OUTPUT_DIR / "patient_fold_assignment.csv", index=False)
pd.DataFrame(oof_rows).to_csv(OUTPUT_DIR / "oof_patient_predictions.csv", index=False)

auc = fold_results["roc_auc"].to_numpy(float)
summary = pd.DataFrame([{
    "study": "Current EfficientNet-B1 implementation",
    "dataset": "BraTS 2021 Task 1 patients matched to MGMT labels",
    "modality": "FLAIR",
    "target": "MGMT promoter methylation",
    "prediction_unit": "patient",
    "n_patients": len(manifest),
    "n_folds": N_SPLITS,
    "mean_roc_auc": float(np.mean(auc)),
    "variance_roc_auc_sample_ddof1": float(np.var(auc, ddof=1)),
    "sd_roc_auc_sample_ddof1": float(np.std(auc, ddof=1)),
    "mean_pr_auc": float(fold_results["pr_auc"].mean()),
}])
summary.to_csv(OUTPUT_DIR / "efficientnet_b1_flair_5fold_summary.csv", index=False)

display(fold_results)
display(summary)

print("Saeed mean from five published folds:", SAEED_AUC_FOLDS.mean())
print("Saeed sample variance ddof=1:", SAEED_AUC_FOLDS.var(ddof=1))


## 9. Saeed-style comparison table

Unlike the old XGBoost/GLCM notebook, this comparison now uses the **same prediction target and prediction unit**: patient-level MGMT classification from FLAIR. However, exact numerical reproduction is still not guaranteed because Saeed et al. do not provide the exact fold membership or every EfficientNet-B1 Table 4 hyperparameter.


In [ ]:
comparison = pd.DataFrame([
    {
        "Study": "This notebook",
        "Model": "3D EfficientNet-B1",
        "Input": "BraTS 2021 Task 1 FLAIR",
        "Target": "MGMT",
        "Validation": "5-fold patient-level StratifiedKFold",
        "Fold 1 AUC": auc[0],
        "Fold 2 AUC": auc[1],
        "Fold 3 AUC": auc[2],
        "Fold 4 AUC": auc[3],
        "Fold 5 AUC": auc[4],
        "Mean AUC": auc.mean(),
        "Variance AUC (ddof=1)": np.var(auc, ddof=1),
    },
    {
        "Study": "Saeed et al. (2023), Table 4",
        "Model": "EfficientNet-B1",
        "Input": "BraTS 2021 Task 1 FLAIR",
        "Target": "MGMT",
        "Validation": "5-fold CV; exact fold assignment not published",
        "Fold 1 AUC": SAEED_AUC_FOLDS[0],
        "Fold 2 AUC": SAEED_AUC_FOLDS[1],
        "Fold 3 AUC": SAEED_AUC_FOLDS[2],
        "Fold 4 AUC": SAEED_AUC_FOLDS[3],
        "Fold 5 AUC": SAEED_AUC_FOLDS[4],
        "Mean AUC": SAEED_MEAN_AUC,
        "Variance AUC (ddof=1)": SAEED_VARIANCE_AUC,
    },
])

comparison.to_csv(OUTPUT_DIR / "comparison_with_saeed_table4.csv", index=False)
display(comparison)


## 10. ROC curve from out-of-fold patient predictions

This curve pools the one out-of-fold prediction produced for each patient. Fold-specific AUCs remain the primary quantities for comparison with Table 4.


In [ ]:
from sklearn.metrics import roc_curve

oof = pd.DataFrame(oof_rows)
if len(oof):
    pooled_auc = roc_auc_score(oof["mgmt_label"], oof["prob_methylated"])
    fpr, tpr, _ = roc_curve(oof["mgmt_label"], oof["prob_methylated"])

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"OOF ROC-AUC = {pooled_auc:.4f}")
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("EfficientNet-B1 + FLAIR — OOF patient-level ROC")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "oof_roc_curve.png", dpi=180)
    plt.show()
    print("Pooled OOF patient-level AUC:", pooled_auc)


# Interpretation rules for the thesis

1. Report the model as **EfficientNet-B1 + FLAIR for patient-level MGMT classification**.
2. Report **five fold AUCs + mean + variance**, not voxel-level metrics.
3. State that the implementation follows Saeed et al. on the elements explicitly reported in the 2023 paper: BraTS 2021 Task 1 subset with MGMT labels, FLAIR, EfficientNet-B1, MONAI/PyTorch, AdamW and 5-fold CV.
4. Clearly label LR, batch size, input size, epoch count, weight decay, z-score normalization, scheduler, seed and exact fold split as **implementation choices** unless independently verified from source code.
5. Do not claim exact reproduction of Saeed et al. solely because the architecture name matches; their exact fold assignment and complete Table 4 hyperparameter set are not published in the paper.
